In [ ]:
import cellestial as cl
import scanpy as sc
from lets_plot import *

LetsPlot.setup_html()

data = sc.read("data/pbmc3k_mini.h5ad")

In [ ]:
data.obs

In [ ]:
data

In [ ]:
from __future__ import annotations

from collections.abc import Iterable
from typing import TYPE_CHECKING

# Core scverse libraries
import polars as pl

# Data retrieval
import scanpy as sc
from lets_plot import LetsPlot, aes, geom_jitter, geom_violin, gggrid, ggplot, layer_tooltips
from lets_plot.plot.core import PlotSpec
from scanpy import AnnData

from cellestial.themes import _THEME_VIOLIN
from cellestial.util import interactive
from math import ceil

LetsPlot.setup_html()

if TYPE_CHECKING:
    from lets_plot.plot.core import PlotSpec


@interactive
def violin(
    data: AnnData,
    key: str,
    *,
    color: str | None = None,
    fill: str | None = None,
    violin_fill: str = "#FF00FF",
    violin_color: str = "#2f2f2f",
    point_color: str = "#1f1f1f",
    point_alpha: float = 0.7,
    point_size: float = 0.5,
    trim: bool = False,
    show_tooltips: bool = True,
    show_points: bool = True,
    add_tooltips: list[str] | tuple[str] | Iterable[str] | None = None,
    custom_tooltips: list[str] | tuple[str] | Iterable[str] | None = None,
    interactive: bool = False,
) -> PlotSpec:
    # check if data is an AnnData object
    if not isinstance(data, sc.AnnData):
        msg = "data must be an AnnData object"
        raise TypeError(msg)
    else:
        frame = pl.from_pandas(data.obs, include_index=True).rename({"None": "CellID"})
    # check if key is in the columns
    if key not in frame.columns:
        msg = f"key must be a column in the AnnData object, but {key} is not in the columns"
        raise KeyError(msg)

    # handle tooltips
    base_tooltips = ["CellID", key]
    if not show_tooltips:
        tooltips = "none"  # for letsplot, this removes the tooltips
    else:
        if isinstance(custom_tooltips, Iterable):
            tooltips = list(custom_tooltips)
        elif isinstance(add_tooltips, Iterable):
            tooltips = base_tooltips + list(add_tooltips)
        else:
            tooltips = base_tooltips

    # handle fill and color
    violin_fill = None if fill is not None else violin_fill
    violin_color = None if color is not None else violin_color
    # handle violimn tooltips
    violin_tooltips = [key]
    violin_tooltips.append(color) if color is not None else None
    violin_tooltips.append(fill) if fill is not None else None
    # generate the plot
    vln = (
        ggplot(data=frame)
        + geom_violin(
            data=frame,
            mapping=aes(y=key, color=color, fill=fill),
            fill=violin_fill,
            color=violin_color,
            trim=trim,
            tooltips=layer_tooltips(violin_tooltips),
        )
        + _THEME_VIOLIN
    )
    # handle the point (jitter)
    if show_points:
        vln += geom_jitter(
            data=frame,
            mapping=aes(y=key),
            color=point_color,
            alpha=point_alpha,
            size=point_size,
            tooltips=layer_tooltips(tooltips),
        )

    # wrap the legend
    if fill is not None:
        n_distinct = frame.select(fill).unique().height
        if n_distinct > 10:
            ncol = ceil(n_distinct / 10)
            vln = vln + guides(fill=guide_legend(ncol=ncol))
    if color is not None:
        n_distinct = frame.select(color).unique().height
        if n_distinct > 10:
            ncol = ceil(n_distinct / 10)
            vln = vln + guides(color=guide_legend(ncol=ncol))

    return vln


@interactive
def violins(
    data,
    keys: list[str] | tuple[str] | Iterable[str],
    *,
    color: str | None = None,
    fill: str | None = None,
    violin_fill: str = "#FF00FF",
    violin_color: str = "#2f2f2f",
    point_color: str = "#1f1f1f",
    point_alpha: float = 0.7,
    point_size: float = 0.5,
    trim: bool = False,
    show_tooltips: bool = True,
    show_points: bool = True,
    add_tooltips: list[str] | tuple[str] | Iterable[str] | None = None,
    custom_tooltips: list[str] | tuple[str] | Iterable[str] | None = None,
    layers: list[str] | tuple[str] | Iterable[str] | None = None,
    interactive: bool = False,
    multi_panel: bool = True,
    **grid_kwargs,
):
    if multi_panel: # standard grid plotting
        plots = list()
        for key in keys:
            vln = violin(
                data,
                key=key,
                color=color,
                fill=fill,
                violin_fill=violin_fill,
                violin_color=violin_color,
                point_color=point_color,
                point_alpha=point_alpha,
                point_size=point_size,
                trim=trim,
                show_tooltips=show_tooltips,
                show_points=show_points,
                add_tooltips=add_tooltips,
                custom_tooltips=custom_tooltips,
                interactive=interactive,
            )
            #handle the layers
            if layers is not None:
                for layer in layers:
                    vln += layer

            plots.append(vln)

        vlns = gggrid(plots, ncol=grid_kwargs.get("ncol"))

    else:  # unpivot the data so that it can be plotted in a single (combined) panel
        frame = pl.from_pandas(data.obs[keys], include_index=True).rename({"None": "CellID"})
        frame = frame.unpivot(index="CellID", variable_name="observations", value_name="value")
        vlns = (
            ggplot(data=frame)
            + geom_violin(aes(x="observations", y="value", fill="observations"))
            + _THEME_VIOLIN
            + ggsize(800, 400)
        )

        #handle the layers
        if layers is not None:
            for layer in layers:
                vlns += layer

    return vlns


In [ ]:
base = violin(data, "n_genes_by_counts", fill="sample", violin_color="#3f3f3f",show_points=False)

In [ ]:
base + labs(title="Cell type distribution", x="Samples", y = "Number of Genes by Counts" ) +ggsize(500, 400)

In [ ]:
violin(data, "n_genes_by_counts", fill="leiden",)+ggsize(800,400) + theme(
    title=element_text(family="Arial", color="#3f3f3f"),)

In [ ]:
data.obs.columns

In [ ]:
violins(
    data,
    ["n_genes_by_counts", "pct_counts_in_top_100_genes", "log1p_total_counts_mt","pct_counts_hb"],
    ncol=2,
    fill="sample",
    show_points=False,
    layers=[scale_y_log10()],
)